# GP02 — Clean Geospatial Data

Κατανοώ → Εντοπίζω → Διορθώνω → Επαληθεύω → Εξάγω

In [ ]:
import pandas as pd
import geopandas as gpd
df = pd.read_csv("data/gp02/observations_dirty.csv")
print("Αρχικές γραμμές:", len(df))
print(df.isna().sum())

In [ ]:
for column in ["longitude", "latitude", "elevation_m"]:
    df[column] = pd.to_numeric(df[column], errors="coerce")
clean = df.dropna(subset=["observation_id", "longitude", "latitude"]).copy()
clean = clean.drop_duplicates(subset="observation_id", keep="first")
clean = clean[clean["longitude"].between(-180, 180) & clean["latitude"].between(-90, 90)].copy()
print("Μετά τον καθαρισμό:", len(clean))

In [ ]:
gdf = gpd.GeoDataFrame(clean, geometry=gpd.points_from_xy(clean["longitude"], clean["latitude"]), crs="EPSG:4326")
print(gdf.crs)
print(gdf.total_bounds)
gdf.plot(figsize=(7, 6), color="darkgreen")

In [ ]:
assert len(gdf) > 0
assert gdf.crs.to_string() == "EPSG:4326"
assert gdf.geometry.geom_type.eq("Point").all()
assert gdf.geometry.notna().all()
assert gdf.drop(columns="geometry").isna().sum().sum() == 0
assert gdf.geometry.is_empty.sum() == 0
assert gdf.geometry.is_valid.all()
assert gdf["observation_id"].is_unique
assert gdf["longitude"].between(-180, 180).all()
assert gdf["latitude"].between(-90, 90).all()
print("Όλα τα QA checks πέρασαν.")

## Mini Challenge — Ενδεικτική λύση

In [ ]:
selected = gdf[(gdf["status"] == "active") & (gdf["elevation_m"] > 80)].copy()
assert len(selected) > 0
assert (selected["elevation_m"] > 80).all()
selected.plot(figsize=(7, 6), color="orange")

## Τελικός έλεγχος

Το καθαρό dataset έχει μοναδικά IDs, valid Points και CRS EPSG:4326.